In [1]:
import os
import json
import shutil
from pathlib import Path
from collections import Counter

import pandas as pd

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI

In [2]:
PROJECT_ROOT = Path(r"C:\Users\asguug\Documents\rag-agent")

if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DOCS_DIR = PROJECT_ROOT / "data" / "docs"
EVAL_DIR = PROJECT_ROOT / "data" / "eval"
CHROMA_WEEK4_DIR = PROJECT_ROOT / "data" / "chroma_week4"

EVAL_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_WEEK4_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DOCS_DIR:", DOCS_DIR)
print("EVAL_DIR:", EVAL_DIR)
print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))

PROJECT_ROOT: C:\Users\asguug\Documents\rag-agent
DOCS_DIR: C:\Users\asguug\Documents\rag-agent\data\docs
EVAL_DIR: C:\Users\asguug\Documents\rag-agent\data\eval
OPENAI_API_KEY exists: True


In [3]:
md_files = sorted(DOCS_DIR.glob("*.md"))

print("Markdown 문서 수:", len(md_files))

for path in md_files:
    print("-", path.name)

assert md_files, f"Markdown 문서가 없습니다. 경로 확인 필요: {DOCS_DIR}"


raw_docs = []

for path in md_files:
    raw_docs.append(
        Document(
            page_content=path.read_text(encoding="utf-8"),
            metadata={
                "source": str(path),
                "source_file": path.name,
                "game_key": path.stem,
                "document_type": "game_profile",
                "source_type": "steam",
            },
        )
    )

print("로드된 문서 수:", len(raw_docs))

Markdown 문서 수: 5
- baldurs_gate_3.md
- cyberpunk_2077.md
- hollow_knight.md
- monster_hunter_world.md
- no_mans_sky.md
로드된 문서 수: 5


In [4]:
def split_markdown_by_fixed_sections(doc: Document) -> list[Document]:
    """
    3주차 baseline에서 사용한 5개 섹션 구조를 재현한다.
    각 Markdown 문서를 metadata / store_summary / about / review / news 섹션으로 나눈다.
    """
    source = doc.metadata.get("source", "")
    source_file = doc.metadata.get("source_file", Path(source).name)
    game_key = doc.metadata.get("game_key", Path(source_file).stem)

    text = doc.page_content

    section_patterns = [
        ("metadata", "## Metadata"),
        ("store_summary", "## Store Summary"),
        ("about", "## About The Game"),
        ("review", "## Recent Steam Reviews"),
        ("news", "## Steam News and Updates"),
    ]

    section_docs = []

    for idx, (section_name, heading) in enumerate(section_patterns):
        start = text.find(heading)

        if start == -1:
            continue

        if idx + 1 < len(section_patterns):
            next_heading = section_patterns[idx + 1][1]
            end = text.find(next_heading, start + len(heading))

            if end == -1:
                end = len(text)
        else:
            end = len(text)

        section_text = text[start:end].strip()

        if not section_text:
            continue

        section_docs.append(
            Document(
                page_content=section_text,
                metadata={
                    **doc.metadata,
                    "source": source,
                    "source_file": source_file,
                    "game_key": game_key,
                    "section": section_name,
                    "document_type": "game_profile",
                    "source_type": "steam",
                },
            )
        )

    return section_docs


section_docs = []

for doc in raw_docs:
    section_docs.extend(split_markdown_by_fixed_sections(doc))

print("원본 Markdown 문서 수:", len(raw_docs))
print("섹션 문서 수:", len(section_docs))

for doc in section_docs[:5]:
    print(doc.metadata["source_file"], doc.metadata["section"], len(doc.page_content))

원본 Markdown 문서 수: 5
섹션 문서 수: 25
baldurs_gate_3.md metadata 636
baldurs_gate_3.md store_summary 224
baldurs_gate_3.md about 5376
baldurs_gate_3.md review 5101
baldurs_gate_3.md news 2704


In [5]:
def infer_game_key_from_source(source: str) -> str:
    return Path(source).stem


def normalize_section_name(section_title: str) -> str:
    """
    Markdown 헤더명 또는 기존 section 값을 검색용 section 값으로 정규화한다.
    """
    section_title = str(section_title).strip().lower()

    if section_title in ["metadata", "store_summary", "about", "review", "news"]:
        return section_title

    if "metadata" in section_title:
        return "metadata"

    if "store summary" in section_title:
        return "store_summary"

    if "about" in section_title:
        return "about"

    if "review" in section_title:
        return "review"

    if "news" in section_title or "update" in section_title:
        return "news"

    return "unknown"


def add_chunk_metadata(
    chunks: list[Document],
    strategy_name: str,
    chunk_size,
    chunk_overlap,
) -> list[Document]:
    """
    chunk별 공통 metadata 보강.
    기존 metadata를 유지하면서 실험 전략 정보를 추가한다.
    """
    enriched = []

    for idx, chunk in enumerate(chunks):
        metadata = dict(chunk.metadata)

        metadata["chunk_strategy"] = strategy_name
        metadata["chunk_index"] = idx
        metadata["chunk_size_setting"] = str(chunk_size)
        metadata["chunk_overlap_setting"] = str(chunk_overlap)
        metadata["char_count"] = len(chunk.page_content)

        if "source_file" not in metadata:
            metadata["source_file"] = Path(metadata.get("source", "")).name

        if "game_key" not in metadata:
            metadata["game_key"] = infer_game_key_from_source(metadata.get("source_file", ""))

        if "section" not in metadata:
            section_title = metadata.get("section_title", "")
            metadata["section"] = normalize_section_name(section_title)
        else:
            metadata["section"] = normalize_section_name(metadata["section"])

        if "document_type" not in metadata:
            metadata["document_type"] = "game_profile"

        if "source_type" not in metadata:
            metadata["source_type"] = "steam"

        enriched.append(
            Document(
                page_content=chunk.page_content,
                metadata=metadata,
            )
        )

    return enriched

In [6]:
def make_chunks_strategy_a(section_docs: list[Document]) -> list[Document]:
    """
    전략 A:
    3주차 baseline 설정 그대로.
    이미 분리된 section_docs에 RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)를 적용한다.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=120,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    chunks = splitter.split_documents(section_docs)

    return add_chunk_metadata(
        chunks=chunks,
        strategy_name="A_recursive_baseline_800_120",
        chunk_size=800,
        chunk_overlap=120,
    )


chunks_a = make_chunks_strategy_a(section_docs)

print("전략 A chunk 수:", len(chunks_a))
print(chunks_a[0].metadata)
print(chunks_a[0].page_content[:500])

전략 A chunk 수: 115
{'source': 'C:\\Users\\asguug\\Documents\\rag-agent\\data\\docs\\baldurs_gate_3.md', 'source_file': 'baldurs_gate_3.md', 'game_key': 'baldurs_gate_3', 'document_type': 'game_profile', 'source_type': 'steam', 'section': 'metadata', 'chunk_strategy': 'A_recursive_baseline_800_120', 'chunk_index': 0, 'chunk_size_setting': '800', 'chunk_overlap_setting': '120', 'char_count': 636}
## Metadata
- game_key: baldurs_gate_3
- appid: 1086940
- title: Baldur's Gate 3
- release_date: Aug 3, 2023
- developers: Larian Studios
- publishers: Larian Studios
- genres: Adventure, RPG, Strategy
- categories: Single-player, Multi-player, Co-op, Online Co-op, LAN Co-op, Cross-Platform Multiplayer, Steam Achievements, Full controller support, Steam Trading Cards, Adjustable Text Size, Camera Comfort, Color Alternatives, Custom Volume Controls, Adjustable Difficulty, Playable without Timed I


In [7]:
def make_chunks_strategy_b(section_docs: list[Document]) -> list[Document]:
    """
    전략 B:
    3주차와 같은 section 분리 방식은 유지하되,
    chunk_size와 chunk_overlap만 변경한다.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    chunks = splitter.split_documents(section_docs)

    return add_chunk_metadata(
        chunks=chunks,
        strategy_name="B_recursive_small_500_100",
        chunk_size=1000,
        chunk_overlap=200,
    )


chunks_b = make_chunks_strategy_b(section_docs)

print("전략 B chunk 수:", len(chunks_b))
print(chunks_b[0].metadata)
print(chunks_b[0].page_content[:500])

전략 B chunk 수: 95
{'source': 'C:\\Users\\asguug\\Documents\\rag-agent\\data\\docs\\baldurs_gate_3.md', 'source_file': 'baldurs_gate_3.md', 'game_key': 'baldurs_gate_3', 'document_type': 'game_profile', 'source_type': 'steam', 'section': 'metadata', 'chunk_strategy': 'B_recursive_small_500_100', 'chunk_index': 0, 'chunk_size_setting': '1000', 'chunk_overlap_setting': '200', 'char_count': 636}
## Metadata
- game_key: baldurs_gate_3
- appid: 1086940
- title: Baldur's Gate 3
- release_date: Aug 3, 2023
- developers: Larian Studios
- publishers: Larian Studios
- genres: Adventure, RPG, Strategy
- categories: Single-player, Multi-player, Co-op, Online Co-op, LAN Co-op, Cross-Platform Multiplayer, Steam Achievements, Full controller support, Steam Trading Cards, Adjustable Text Size, Camera Comfort, Color Alternatives, Custom Volume Controls, Adjustable Difficulty, Playable without Timed I


In [8]:
def make_chunks_strategy_c(raw_docs: list[Document]) -> list[Document]:
    """
    전략 C:
    MarkdownHeaderTextSplitter로 Markdown 헤더 구조를 먼저 보존한 뒤,
    길이가 긴 header chunk만 RecursiveCharacterTextSplitter로 추가 분할한다.
    """
    headers_to_split_on = [
        ("#", "title"),
        ("##", "section_title"),
        ("###", "subsection_title"),
    ]

    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=False,
    )

    header_docs = []

    for doc in raw_docs:
        source = doc.metadata.get("source", "")
        source_file = doc.metadata.get("source_file", Path(source).name)
        game_key = doc.metadata.get("game_key", infer_game_key_from_source(source))

        md_splits = markdown_splitter.split_text(doc.page_content)

        for split_doc in md_splits:
            section_title = split_doc.metadata.get("section_title", "")
            section = normalize_section_name(section_title)

            header_docs.append(
                Document(
                    page_content=split_doc.page_content,
                    metadata={
                        **doc.metadata,
                        **split_doc.metadata,
                        "source": source,
                        "source_file": source_file,
                        "game_key": game_key,
                        "section": section,
                        "document_type": "game_profile",
                        "source_type": "steam",
                    },
                )
            )

    # 너무 긴 header split만 다시 recursive split
    recursive_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=120,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    chunks = recursive_splitter.split_documents(header_docs)

    return add_chunk_metadata(
        chunks=chunks,
        strategy_name="C_markdown_header_recursive_800_120",
        chunk_size="header_then_800",
        chunk_overlap=120,
    )


chunks_c = make_chunks_strategy_c(raw_docs)

print("전략 C chunk 수:", len(chunks_c))
print(chunks_c[0].metadata)
print(chunks_c[0].page_content[:500])

전략 C chunk 수: 184
{'source': 'C:\\Users\\asguug\\Documents\\rag-agent\\data\\docs\\baldurs_gate_3.md', 'source_file': 'baldurs_gate_3.md', 'game_key': 'baldurs_gate_3', 'document_type': 'game_profile', 'source_type': 'steam', 'title': "Baldur's Gate 3", 'section_title': 'Metadata', 'section': 'metadata', 'chunk_strategy': 'C_markdown_header_recursive_800_120', 'chunk_index': 0, 'chunk_size_setting': 'header_then_800', 'chunk_overlap_setting': '120', 'char_count': 656}
# Baldur's Gate 3  
## Metadata
- game_key: baldurs_gate_3
- appid: 1086940
- title: Baldur's Gate 3
- release_date: Aug 3, 2023
- developers: Larian Studios
- publishers: Larian Studios
- genres: Adventure, RPG, Strategy
- categories: Single-player, Multi-player, Co-op, Online Co-op, LAN Co-op, Cross-Platform Multiplayer, Steam Achievements, Full controller support, Steam Trading Cards, Adjustable Text Size, Camera Comfort, Color Alternatives, Custom Volume Controls, Adjustable Difficulty, Play


In [9]:
def remove_related_links(text: str) -> str:
    """
    Steam News 안의 RELATED LINKS 이후 부가 링크 텍스트를 제거한다.
    뉴스 본문보다 관련 기사 링크 제목이 검색에 걸리는 문제를 줄이기 위함.
    """
    markers = [
        "RELATED LINKS:",
        "Read the rest of the story...",
    ]

    for marker in markers:
        if marker in text:
            text = text.split(marker)[0].strip()

    return text


def make_chunks_strategy_d(raw_docs: list[Document]) -> list[Document]:
    """
    전략 D:
    Markdown 구조는 보존하되, # / ## 헤더까지만 사용한다.
    ### Review, ### News 단위까지는 쪼개지 않는다.
    이후 긴 섹션은 RecursiveCharacterTextSplitter(1000/200)로 분할한다.
    """
    headers_to_split_on = [
        ("#", "title"),
        ("##", "section_title"),
    ]

    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=False,
    )

    header_docs = []

    for doc in raw_docs:
        source = doc.metadata.get("source", "")
        source_file = doc.metadata.get("source_file", Path(source).name)
        game_key = doc.metadata.get("game_key", infer_game_key_from_source(source))

        cleaned_text = remove_related_links(doc.page_content)
        md_splits = markdown_splitter.split_text(cleaned_text)

        for split_doc in md_splits:
            section_title = split_doc.metadata.get("section_title", "")
            section = normalize_section_name(section_title)

            header_docs.append(
                Document(
                    page_content=split_doc.page_content,
                    metadata={
                        **doc.metadata,
                        **split_doc.metadata,
                        "source": source,
                        "source_file": source_file,
                        "game_key": game_key,
                        "section": section,
                        "document_type": "game_profile",
                        "source_type": "steam",
                    },
                )
            )

    recursive_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    chunks = recursive_splitter.split_documents(header_docs)

    return add_chunk_metadata(
        chunks=chunks,
        strategy_name="D_markdown_h2_recursive_1000_200",
        chunk_size="h2_then_1000",
        chunk_overlap=200,
    )


chunks_d = make_chunks_strategy_d(raw_docs)

print("전략 D chunk 수:", len(chunks_d))
print(chunks_d[0].metadata)
print(chunks_d[0].page_content[:500])

전략 D chunk 수: 89
{'source': 'C:\\Users\\asguug\\Documents\\rag-agent\\data\\docs\\baldurs_gate_3.md', 'source_file': 'baldurs_gate_3.md', 'game_key': 'baldurs_gate_3', 'document_type': 'game_profile', 'source_type': 'steam', 'title': "Baldur's Gate 3", 'section_title': 'Metadata', 'section': 'metadata', 'chunk_strategy': 'D_markdown_h2_recursive_1000_200', 'chunk_index': 0, 'chunk_size_setting': 'h2_then_1000', 'chunk_overlap_setting': '200', 'char_count': 656}
# Baldur's Gate 3  
## Metadata
- game_key: baldurs_gate_3
- appid: 1086940
- title: Baldur's Gate 3
- release_date: Aug 3, 2023
- developers: Larian Studios
- publishers: Larian Studios
- genres: Adventure, RPG, Strategy
- categories: Single-player, Multi-player, Co-op, Online Co-op, LAN Co-op, Cross-Platform Multiplayer, Steam Achievements, Full controller support, Steam Trading Cards, Adjustable Text Size, Camera Comfort, Color Alternatives, Custom Volume Controls, Adjustable Difficulty, Play


In [10]:
strategy_chunks = {
    "A_recursive_baseline_800_120": chunks_a,
    "B_recursive_large_1000_200": chunks_b,
    "C_markdown_header_recursive_800_120": chunks_c,
    "D_markdown_h2_recursive_1000_200": chunks_d,
}

chunk_stat_records = []

for strategy_name, chunks in strategy_chunks.items():
    lengths = [len(chunk.page_content) for chunk in chunks]
    section_counter = Counter([chunk.metadata.get("section", "unknown") for chunk in chunks])

    chunk_stat_records.append(
        {
            "strategy": strategy_name,
            "chunk_count": len(chunks),
            "min_chars": min(lengths),
            "max_chars": max(lengths),
            "avg_chars": round(sum(lengths) / len(lengths), 1),
            "metadata_fields_example": sorted(list(chunks[0].metadata.keys())),
            "section_distribution": json.dumps(dict(section_counter), ensure_ascii=False),
        }
    )

chunk_stats_df = pd.DataFrame(chunk_stat_records)

display(chunk_stats_df)

chunk_stats_path = EVAL_DIR / "week4_chunk_stats.csv"
chunk_stats_df.to_csv(chunk_stats_path, index=False, encoding="utf-8-sig")

print("saved:", chunk_stats_path)

,strategy,chunk_count,min_chars,max_chars,avg_chars,metadata_fields_example,section_distribution
0,A_recursive_baseline_800_120,115,62,799,569.4,"[char_count, chunk_index, chunk_overlap_settin...","{""metadata"": 5, ""store_summary"": 5, ""about"": 2..."
1,B_recursive_large_1000_200,95,62,999,707.5,"[char_count, chunk_index, chunk_overlap_settin...","{""metadata"": 5, ""store_summary"": 5, ""about"": 2..."
2,C_markdown_header_recursive_800_120,184,62,799,342.1,"[char_count, chunk_index, chunk_overlap_settin...","{""metadata"": 5, ""store_summary"": 5, ""about"": 2..."
3,D_markdown_h2_recursive_1000_200,89,62,999,724.7,"[char_count, chunk_index, chunk_overlap_settin...","{""metadata"": 5, ""store_summary"": 5, ""about"": 2..."


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week4_chunk_stats.csv


In [11]:
for strategy_name, chunks in strategy_chunks.items():
    print("\n" + "=" * 100)
    print(strategy_name)

    for i, chunk in enumerate(chunks[:2], start=1):
        print(f"\n--- chunk {i} metadata ---")
        print(chunk.metadata)
        print(chunk.page_content[:300].replace("\n", " "))


A_recursive_baseline_800_120

--- chunk 1 metadata ---
{'source': 'C:\\Users\\asguug\\Documents\\rag-agent\\data\\docs\\baldurs_gate_3.md', 'source_file': 'baldurs_gate_3.md', 'game_key': 'baldurs_gate_3', 'document_type': 'game_profile', 'source_type': 'steam', 'section': 'metadata', 'chunk_strategy': 'A_recursive_baseline_800_120', 'chunk_index': 0, 'chunk_size_setting': '800', 'chunk_overlap_setting': '120', 'char_count': 636}
## Metadata - game_key: baldurs_gate_3 - appid: 1086940 - title: Baldur's Gate 3 - release_date: Aug 3, 2023 - developers: Larian Studios - publishers: Larian Studios - genres: Adventure, RPG, Strategy - categories: Single-player, Multi-player, Co-op, Online Co-op, LAN Co-op, Cross-Platform Multipla

--- chunk 2 metadata ---
{'source': 'C:\\Users\\asguug\\Documents\\rag-agent\\data\\docs\\baldurs_gate_3.md', 'source_file': 'baldurs_gate_3.md', 'game_key': 'baldurs_gate_3', 'document_type': 'game_profile', 'source_type': 'steam', 'section': 'store_summary', 'c

In [12]:
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [13]:
def build_vectorstore(strategy_name: str, chunks: list[Document]) -> Chroma:
    persist_dir = CHROMA_WEEK4_DIR / strategy_name

    if persist_dir.exists():
        shutil.rmtree(persist_dir)

    persist_dir.mkdir(parents=True, exist_ok=True)

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=strategy_name,
        persist_directory=str(persist_dir),
    )

    return vectorstore


vectorstores = {}

for strategy_name, chunks in strategy_chunks.items():
    print("Building vectorstore:", strategy_name)
    vectorstores[strategy_name] = build_vectorstore(strategy_name, chunks)
    print("done:", strategy_name, "chunk_count:", len(chunks))

Building vectorstore: A_recursive_baseline_800_120
done: A_recursive_baseline_800_120 chunk_count: 115
Building vectorstore: B_recursive_large_1000_200
done: B_recursive_large_1000_200 chunk_count: 95
Building vectorstore: C_markdown_header_recursive_800_120
done: C_markdown_header_recursive_800_120 chunk_count: 184
Building vectorstore: D_markdown_h2_recursive_1000_200
done: D_markdown_h2_recursive_1000_200 chunk_count: 89


In [14]:
llm = ChatOpenAI(
    model="gpt-5-mini",
    # temperature 에러가 나면 temperature 인자를 제거한 현재 설정을 유지한다.
)

GAME_ALIASES = {
    "hollow_knight": [
        "hollow knight", "할로우 나이트", "할로우나이트"
    ],
    "monster_hunter_world": [
        "monster hunter: world", "monster hunter world",
        "몬스터 헌터 월드", "몬헌 월드", "몬헌월드"
    ],
    "baldurs_gate_3": [
        "baldur's gate 3", "baldurs gate 3",
        "발더스 게이트 3", "발더스3", "발더스 게이트"
    ],
    "no_mans_sky": [
        "no man's sky", "no mans sky",
        "노 맨즈 스카이", "노맨즈스카이"
    ],
    "cyberpunk_2077": [
        "cyberpunk 2077", "사이버펑크 2077",
        "사펑", "사이버펑크"
    ],
}


def detect_game_key(query: str):
    query_lower = query.lower()

    for game_key, aliases in GAME_ALIASES.items():
        for alias in aliases:
            if alias.lower() in query_lower:
                return game_key

    return None


def detect_intent(query: str):
    query_lower = query.lower()

    review_keywords = [
        "review", "reviews", "recent review", "steam review",
        "리뷰", "최근 리뷰", "평가", "반응", "유저 반응", "민심"
    ]

    news_keywords = [
        "update", "updates", "patch", "news", "recent update",
        "업데이트", "패치", "뉴스", "변경", "개선"
    ]

    gameplay_keywords = [
        "play style", "gameplay", "core loop", "combat", "style",
        "플레이 스타일", "플레이 루프", "핵심 플레이", "핵심 루프",
        "전투", "조작", "방식", "특징"
    ]

    if any(keyword in query_lower for keyword in review_keywords):
        return "review"

    if any(keyword in query_lower for keyword in news_keywords):
        return "news"

    if any(keyword in query_lower for keyword in gameplay_keywords):
        return "gameplay"

    return "general"

def build_filter(intent: str, game_key: str | None):
    conditions = []

    if intent == "review":
        conditions.append({"section": {"$eq": "review"}})
    elif intent == "news":
        conditions.append({"section": {"$eq": "news"}})

    if game_key is not None:
        conditions.append({"game_key": {"$eq": game_key}})

    if len(conditions) == 0:
        return None

    if len(conditions) == 1:
        return conditions[0]

    return {"$and": conditions}


rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a Steam game recommendation and analysis assistant.

Answer the user's question using only the provided context.
If the context is insufficient, say that the available document does not contain enough evidence.
Do not invent details that are not supported by the context.

Write the answer in Korean.
Keep the answer concise but grounded.
            """.strip(),
        ),
        (
            "human",
            """
[Question]
{question}

[Context]
{context}
            """.strip(),
        ),
    ]
)


def format_docs(docs):
    formatted = []

    for i, doc in enumerate(docs, start=1):
        source = Path(doc.metadata.get("source", "")).name
        section = doc.metadata.get("section", "unknown")
        game_key = doc.metadata.get("game_key", "unknown")
        strategy = doc.metadata.get("chunk_strategy", "unknown")
        content = doc.page_content

        formatted.append(
            f"[Context {i}]\n"
            f"source: {source}\n"
            f"game_key: {game_key}\n"
            f"section: {section}\n"
            f"chunk_strategy: {strategy}\n"
            f"{content}"
        )

    return "\n\n".join(formatted)

def rerank_by_section(docs, intent: str):
    if intent == "gameplay":
        priority = {
            "about": 0,
            "store_summary": 1,
            "metadata": 2,
            "review": 3,
            "news": 4,
        }
    elif intent == "review":
        priority = {
            "review": 0,
            "metadata": 1,
            "about": 2,
            "store_summary": 3,
            "news": 4,
        }
    elif intent == "news":
        priority = {
            "news": 0,
            "metadata": 1,
            "review": 2,
            "about": 3,
            "store_summary": 4,
        }
    else:
        priority = {
            "about": 0,
            "store_summary": 1,
            "metadata": 2,
            "review": 3,
            "news": 4,
        }

    return sorted(
        docs,
        key=lambda doc: priority.get(doc.metadata.get("section", "unknown"), 99)
    )

    
def retrieve_docs(vectorstore: Chroma, query: str, k: int = 5):
    intent = detect_intent(query)
    game_key = detect_game_key(query)
    metadata_filter = build_filter(intent=intent, game_key=game_key)

    if metadata_filter is None:
        retrieved_docs = vectorstore.similarity_search(
            query=query,
            k=k,
        )
    else:
        retrieved_docs = vectorstore.similarity_search(
            query=query,
            k=k,
            filter=metadata_filter,
        )

    retrieved_docs = rerank_by_section(retrieved_docs, intent)
    return intent, game_key, metadata_filter, retrieved_docs


def run_rag(vectorstore: Chroma, query: str, k: int = 5):
    intent, game_key, metadata_filter, retrieved_docs = retrieve_docs(
        vectorstore=vectorstore,
        query=query,
        k=k,
    )

    context = format_docs(retrieved_docs)

    messages = rag_prompt.format_messages(
        question=query,
        context=context,
    )

    response = llm.invoke(messages)

    return {
        "question": query,
        "intent": intent,
        "game_key": game_key,
        "metadata_filter": metadata_filter,
        "retrieved_docs": retrieved_docs,
        "answer": response.content,
    }

In [15]:
diagnostic_questions = [
    "Hollow Knight는 어떤 플레이 스타일의 게임인가요?",
    "Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?",
    "No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?",
    "Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?",
]

retrieval_records = []

for strategy_name, vectorstore in vectorstores.items():
    print("\n" + "#" * 120)
    print("Strategy:", strategy_name)

    for question in diagnostic_questions:
        result = run_rag(vectorstore, question)

        print("\n" + "=" * 100)
        print("Question:", question)
        print("Intent:", result["intent"])
        print("Game key:", result["game_key"])
        print("Filter:", result["metadata_filter"])
        print("Answer:", result["answer"][:300])

        for rank, doc in enumerate(result["retrieved_docs"], start=1):
            source = Path(doc.metadata.get("source", "")).name
            section = doc.metadata.get("section", "unknown")
            game_key = doc.metadata.get("game_key", "unknown")
            preview = doc.page_content[:250].replace("\n", " ")

            print(f"[{rank}] source={source} | game_key={game_key} | section={section} | preview={preview}")

            retrieval_records.append(
                {
                    "strategy": strategy_name,
                    "question": question,
                    "intent": result["intent"],
                    "detected_game_key": result["game_key"],
                    "rank": rank,
                    "source": source,
                    "game_key": game_key,
                    "section": section,
                    "preview": preview,
                }
            )

retrieval_df = pd.DataFrame(retrieval_records)

display(retrieval_df.head(30))

retrieval_path = EVAL_DIR / "week4_retrieval_diagnostics.csv"
retrieval_df.to_csv(retrieval_path, index=False, encoding="utf-8-sig")

print("saved:", retrieval_path)


########################################################################################################################
Strategy: A_recursive_baseline_800_120

Question: Hollow Knight는 어떤 플레이 스타일의 게임인가요?
Intent: gameplay
Game key: hollow_knight
Filter: {'game_key': {'$eq': 'hollow_knight'}}
Answer: Hollow Knight는 클래식 스타일의 2D 횡스크롤 액션 어드벤처입니다.  
조작은 정밀하게 튜닝된 2D 컨트롤(회피·대시·베기)을 중심으로 한 전투 위주이고, 광대한 연결형 맵을 탐험하며 길을 개척하고 보스와 적들을 상대하는 플레이가 핵심입니다.  
싱글플레이 중심이며, 도전 요소(예: 클리어 후 해금되는 Steel Soul Mode)와 추가 퀘스트·보스가 포함된 무료 확장팩들로 콘텐츠가 확장됩니다.
[1] source=hollow_knight.md | game_key=hollow_knight | section=about | preview=Hollow Knight is a classically styled 2D action adventure across a vast interconnected world. Explore twisting caverns, ancient cities and deadly wastes; battle tainted creatures and befriend bizarre bugs; and solve ancient mysteries at the kingdom's
[2] source=hollow_knight.md | game_key=hollow_knight | section=about | preview=Complete Hollow Knight to unlock Steel Soul Mode, the ultim

,strategy,question,intent,detected_game_key,rank,source,game_key,section,preview
0,A_recursive_baseline_800_120,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay,hollow_knight,1,hollow_knight.md,hollow_knight,about,Hollow Knight is a classically styled 2D actio...
1,A_recursive_baseline_800_120,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay,hollow_knight,2,hollow_knight.md,hollow_knight,about,Complete Hollow Knight to unlock Steel Soul Mo...
2,A_recursive_baseline_800_120,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay,hollow_knight,3,hollow_knight.md,hollow_knight,about,## About The Game Hollow Knight Expands with F...
3,A_recursive_baseline_800_120,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay,hollow_knight,4,hollow_knight.md,hollow_knight,store_summary,## Store Summary Forge your own path in Hollow...
4,A_recursive_baseline_800_120,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay,hollow_knight,5,hollow_knight.md,hollow_knight,metadata,## Metadata - game_key: hollow_knight - appid:...
5,A_recursive_baseline_800_120,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,gameplay,monster_hunter_world,1,monster_hunter_world.md,monster_hunter_world,about,## About The Game Welcome to a new world! Take...
6,A_recursive_baseline_800_120,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,gameplay,monster_hunter_world,2,monster_hunter_world.md,monster_hunter_world,about,"In Monster Hunter: World, the latest installme..."
7,A_recursive_baseline_800_120,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,gameplay,monster_hunter_world,3,monster_hunter_world.md,monster_hunter_world,store_summary,## Store Summary Welcome to a new world! In Mo...
8,A_recursive_baseline_800_120,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,gameplay,monster_hunter_world,4,monster_hunter_world.md,monster_hunter_world,news,Monster Hunter Stories is an RPG series set in...
9,A_recursive_baseline_800_120,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,gameplay,monster_hunter_world,5,monster_hunter_world.md,monster_hunter_world,news,{STEAM_CLAN_IMAGE}/45725708/73392ac8e0f10e408b...


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week4_retrieval_diagnostics.csv


In [16]:
ragas_questions = [
    "Hollow Knight는 어떤 플레이 스타일의 게임인가요?",
    "No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?",
    "Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?",
]

ragas_references = [
    "Hollow Knight는 2D 사이드스크롤 액션 어드벤처 게임으로, 거대한 연결형 세계를 탐험하고 적과 전투하며 고대 왕국의 비밀을 밝혀가는 플레이 스타일을 가진다.",
    "No Man's Sky는 지속적인 업데이트를 통해 새로운 콘텐츠와 시스템을 추가해 왔으며, 최근 뉴스에서는 새로운 업데이트와 게임 확장 방향이 언급된다.",
    "Cyberpunk 2077의 최근 Steam 리뷰는 대체로 긍정적인 반응이 많으며, 스토리, 캐릭터, 몰입감, 모딩 지원 등에 대한 호평이 확인된다.",
]

len(ragas_questions), len(ragas_references)

(3, 3)

In [17]:
ragas_records = []

for strategy_name, vectorstore in vectorstores.items():
    print("Running RAG for:", strategy_name)

    for question, reference in zip(ragas_questions, ragas_references):
        result = run_rag(vectorstore, question)

        contexts = [
            doc.page_content
            for doc in result["retrieved_docs"]
        ]

        retrieved_sources = [
            {
                "source": Path(doc.metadata.get("source", "")).name,
                "game_key": doc.metadata.get("game_key"),
                "section": doc.metadata.get("section"),
                "chunk_strategy": doc.metadata.get("chunk_strategy"),
            }
            for doc in result["retrieved_docs"]
        ]

        ragas_records.append(
            {
                "strategy": strategy_name,
                "question": question,
                "answer": result["answer"],
                "contexts": contexts,
                "ground_truth": reference,
                "retrieved_sources": json.dumps(retrieved_sources, ensure_ascii=False),

                # 일부 RAGAS 버전 호환용 컬럼
                "user_input": question,
                "response": result["answer"],
                "retrieved_contexts": contexts,
                "reference": reference,
            }
        )

ragas_input_df = pd.DataFrame(ragas_records)

display(ragas_input_df[["strategy", "question", "answer", "ground_truth", "retrieved_sources"]])

ragas_input_path = EVAL_DIR / "week4_ragas_inputs.csv"
ragas_input_df.to_csv(ragas_input_path, index=False, encoding="utf-8-sig")

print("saved:", ragas_input_path)

Running RAG for: A_recursive_baseline_800_120
Running RAG for: B_recursive_large_1000_200
Running RAG for: C_markdown_header_recursive_800_120
Running RAG for: D_markdown_h2_recursive_1000_200


,strategy,question,answer,ground_truth,retrieved_sources
0,A_recursive_baseline_800_120,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,"Hollow Knight는 2D 횡스크롤 액션 어드벤처로, 탐험과 전투가 중심인 플...","Hollow Knight는 2D 사이드스크롤 액션 어드벤처 게임으로, 거대한 연결형...","[{""source"": ""hollow_knight.md"", ""game_key"": ""h..."
1,A_recursive_baseline_800_120,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,문맥에 따르면 최근 No Man's Sky 업데이트는 주로 2026년 4월에 공개된...,No Man's Sky는 지속적인 업데이트를 통해 새로운 콘텐츠와 시스템을 추가해 ...,"[{""source"": ""no_mans_sky.md"", ""game_key"": ""no_..."
2,A_recursive_baseline_800_120,Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?,제공된 최근 Steam 리뷰들은 대체로 매우 긍정적입니다. 반복되는 평은 다음과 같...,"Cyberpunk 2077의 최근 Steam 리뷰는 대체로 긍정적인 반응이 많으며,...","[{""source"": ""cyberpunk_2077.md"", ""game_key"": ""..."
3,B_recursive_large_1000_200,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,Hollow Knight는 “클래식 스타일의 2D 액션 어드벤처” 게임입니다. 주요...,"Hollow Knight는 2D 사이드스크롤 액션 어드벤처 게임으로, 거대한 연결형...","[{""source"": ""hollow_knight.md"", ""game_key"": ""h..."
4,B_recursive_large_1000_200,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,"요약하면 최근 No Man's Sky의 대형 업데이트는 ""Xeno Arena""로, ...",No Man's Sky는 지속적인 업데이트를 통해 새로운 콘텐츠와 시스템을 추가해 ...,"[{""source"": ""no_mans_sky.md"", ""game_key"": ""no_..."
5,B_recursive_large_1000_200,Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?,제공된 최근 Steam 리뷰(모두 2026-05-03 작성)는 대체로 매우 긍정적입...,"Cyberpunk 2077의 최근 Steam 리뷰는 대체로 긍정적인 반응이 많으며,...","[{""source"": ""cyberpunk_2077.md"", ""game_key"": ""..."
6,C_markdown_header_recursive_800_120,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,Hollow Knight는 클래식한 2D 액션 어드벤처(메트로이드배니아 스타일) 게...,"Hollow Knight는 2D 사이드스크롤 액션 어드벤처 게임으로, 거대한 연결형...","[{""source"": ""hollow_knight.md"", ""game_key"": ""h..."
7,C_markdown_header_recursive_800_120,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,간단히 요약하면 최신 업데이트는 XENO ARENA(2026년 4월 발표)입니다.\...,No Man's Sky는 지속적인 업데이트를 통해 새로운 콘텐츠와 시스템을 추가해 ...,"[{""source"": ""no_mans_sky.md"", ""game_key"": ""no_..."
8,C_markdown_header_recursive_800_120,Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?,최근 리뷰들은 대체로 긍정적입니다 (모두 2026-05-03 작성). 주요 특징은 ...,"Cyberpunk 2077의 최근 Steam 리뷰는 대체로 긍정적인 반응이 많으며,...","[{""source"": ""cyberpunk_2077.md"", ""game_key"": ""..."
9,D_markdown_h2_recursive_1000_200,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,Hollow Knight는 클래식한 2D 액션 어드벤처 게임입니다. \n방대한 상...,"Hollow Knight는 2D 사이드스크롤 액션 어드벤처 게임으로, 거대한 연결형...","[{""source"": ""hollow_knight.md"", ""game_key"": ""h..."


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week4_ragas_inputs.csv


In [18]:
from datasets import Dataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
)

ragas_eval_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

evaluator_llm = LangchainLLMWrapper(ragas_eval_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
]

for metric in metrics:
    if hasattr(metric, "llm"):
        metric.llm = evaluator_llm
    if hasattr(metric, "embeddings"):
        metric.embeddings = evaluator_embeddings


ragas_score_records = []

for strategy_name in strategy_chunks.keys():
    print("\n" + "=" * 100)
    print("Evaluating:", strategy_name)

    strategy_df = ragas_input_df[ragas_input_df["strategy"] == strategy_name].copy()

    ragas_dataset = Dataset.from_dict(
        {
            "question": strategy_df["question"].tolist(),
            "answer": strategy_df["answer"].tolist(),
            "contexts": strategy_df["contexts"].tolist(),
            "ground_truth": strategy_df["ground_truth"].tolist(),
        }
    )

    ragas_result = evaluate(
        ragas_dataset,
        metrics=metrics,
    )

    print(ragas_result)

    # RAGAS 버전별 결과 변환 호환 처리
    if hasattr(ragas_result, "to_pandas"):
        result_df = ragas_result.to_pandas()
        mean_scores = result_df.mean(numeric_only=True).to_dict()
    else:
        mean_scores = dict(ragas_result)

    record = {
        "strategy": strategy_name,
        "question_count": len(strategy_df),
    }

    record.update(mean_scores)

    ragas_score_records.append(record)

ragas_scores_df = pd.DataFrame(ragas_score_records)

display(ragas_scores_df)

C:\Users\asguug\AppData\Local\Temp\ipykernel_48456\2777735142.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\asguug\AppData\Local\Temp\ipykernel_48456\2777735142.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\asguug\AppData\Local\Temp\ipykernel_48456\2777735142.py:6: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\asguug\AppData\Local\Temp\ipykerne


Evaluating: A_recursive_baseline_800_120


Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'faithfulness': 0.9667, 'answer_relevancy': 0.6564, 'context_precision': 0.9625}

Evaluating: B_recursive_large_1000_200


...

{'faithfulness': 0.9167, 'answer_relevancy': 0.4428, 'context_precision': 0.9347}

Evaluating: D_markdown_h2_recursive_1000_200


Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'faithfulness': 0.9697, 'answer_relevancy': 0.5431, 'context_precision': 1.0000}


,strategy,question_count,faithfulness,answer_relevancy,context_precision
0,A_recursive_baseline_800_120,3,0.966667,0.656407,0.962500
1,B_recursive_large_1000_200,3,1.000000,0.666869,0.934722
2,C_markdown_header_recursive_800_120,3,0.916667,0.442846,0.934722
3,D_markdown_h2_recursive_1000_200,3,0.969697,0.543138,1.000000


In [19]:
ragas_scores_path = EVAL_DIR / "week4_chunking_ragas_scores.csv"
ragas_scores_df.to_csv(ragas_scores_path, index=False, encoding="utf-8-sig")

print("saved:", ragas_scores_path)

# 3주차 baseline 점수
week3_baseline = {
    "faithfulness": 0.9259,
    "answer_relevancy": 0.4284,
    "context_precision": 1.0000,
}

comparison_df = ragas_scores_df.copy()

for metric, baseline_score in week3_baseline.items():
    if metric in comparison_df.columns:
        comparison_df[f"{metric}_week3_baseline"] = baseline_score
        comparison_df[f"{metric}_delta"] = comparison_df[metric] - baseline_score

display(comparison_df)

comparison_path = EVAL_DIR / "week4_vs_week3_comparison.csv"
comparison_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")

print("saved:", comparison_path)

saved: C:\Users\asguug\Documents\rag-agent\data\eval\week4_chunking_ragas_scores.csv


,strategy,question_count,faithfulness,answer_relevancy,context_precision,faithfulness_week3_baseline,faithfulness_delta,answer_relevancy_week3_baseline,answer_relevancy_delta,context_precision_week3_baseline,context_precision_delta
0,A_recursive_baseline_800_120,3,0.966667,0.656407,0.962500,0.9259,0.040767,0.4284,0.228007,1.0,-3.750000e-02
1,B_recursive_large_1000_200,3,1.000000,0.666869,0.934722,0.9259,0.074100,0.4284,0.238469,1.0,-6.527778e-02
2,C_markdown_header_recursive_800_120,3,0.916667,0.442846,0.934722,0.9259,-0.009233,0.4284,0.014446,1.0,-6.527778e-02
3,D_markdown_h2_recursive_1000_200,3,0.969697,0.543138,1.000000,0.9259,0.043797,0.4284,0.114738,1.0,-3.166667e-11


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week4_vs_week3_comparison.csv
